# ¿Qué es un Agente?

## 1. Definición

Un **agente** es un sistema que usa un modelo de lenguaje (LLM) como su
"cerebro" para **decidir qué acciones tomar**, de forma iterativa, con el
objetivo de resolver una tarea.

La diferencia clave frente a una llamada tradicional a un LLM es esta:

| LLM tradicional | Agente |
|---|---|
| Recibe un prompt → genera texto → termina | Recibe un objetivo → **razona, actúa, observa, repite** hasta cumplirlo |
| No puede consultar el mundo exterior | Puede usar **herramientas** (APIs, bases de datos, funciones) |
| Flujo fijo, decidido por el desarrollador | Flujo dinámico, **decidido por el modelo** en tiempo de ejecución |

> 💡 En pocas palabras: un LLM normal *responde*. Un agente *actúa*.

---

## 2. Componentes de un agente

Todo agente, sin importar el framework, se construye a partir de estas piezas:

### 🧠 a) El modelo (LLM)
Es el motor de razonamiento. No ejecuta acciones directamente, pero decide
**cuál** es la siguiente acción a tomar dado el estado actual.

### 🛠️ b) Herramientas (Tools)
Son funciones que el agente puede invocar para interactuar con el mundo
exterior: consultar una base de datos, llamar una API, ejecutar una
consulta SQL, leer un documento, etc.

```python
# Ejemplo: una herramienta simple en LangChain
from langchain_core.tools import tool

@tool
def consultar_estado_orden(id_orden: str) -> str:
    """Consulta el estado actual de una orden de calidad de facturación."""
    # lógica real aquí
    return "Orden en revisión, pendiente de validación de lectura"
```

### 🔁 c) El ciclo de razonamiento (Reasoning loop)
El agente no responde de una sola vez. Sigue un ciclo (comúnmente llamado
**ReAct**: *Reason + Act*):

1. **Pensar** → ¿qué necesito hacer para resolver esto?
2. **Actuar** → invocar una herramienta
3. **Observar** → recibir el resultado de esa herramienta
4. **Repetir** hasta tener suficiente información para responder

### 💾 d) Memoria (Memory)
Permite que el agente recuerde el contexto de la conversación o de pasos
anteriores dentro de la misma tarea (por ejemplo, qué herramientas ya usó
y qué resultados obtuvo).

### 🧭 e) Orquestación / Estado (State / Graph)
Define **cómo se conectan** los pasos: qué pasa después de cada decisión,
cuándo terminar, cuándo pedir más información. En LangChain moderno esto
se modela con **LangGraph**, como un grafo de nodos y transiciones.

---

## 3. ¿Qué problema resuelve un agente?

Un LLM por sí solo tiene tres limitaciones importantes:

1. **No conoce datos actuales ni privados** (por ejemplo, el estado real
   de una orden de facturación en tu base de datos).
2. **No puede ejecutar acciones** (no puede consultar un sistema, hacer un
   cálculo exacto, o modificar un registro).
3. **Sigue un único paso de razonamiento**, sin posibilidad de corregirse
   con información nueva a mitad de camino.

Un agente resuelve esto dándole al LLM la capacidad de:

- **Consultar información real** en el momento en que la necesita (vía tools).
- **Tomar decisiones multi-paso**, ajustando su plan según lo que va
  observando.
- **Combinar razonamiento flexible (LLM) con lógica determinística**
  (código) cuando la tarea lo requiere — por ejemplo, separar reglas de
  negocio fijas (Python) de tareas de interpretación de lenguaje natural
  (LLM).

> 🎯 Ejemplo aplicado: en lugar de que un analista revise manualmente cada
> orden de calidad de facturación, un agente puede leer la orden,
> consultar el histórico de consumo, clasificar el tipo de error de
> lectura, y decidir si escalarla o cerrarla — usando el LLM solo donde
> realmente aporta valor (interpretar texto libre), y código determinístico
> para las reglas de negocio.

---

## 4. Resumen visual del flujo

```
Objetivo del usuario
        │
        ▼
   ┌─────────┐
   │  LLM    │──── decide acción ────▶ ┌───────────┐
   │(razona) │                         │  Tool     │
   └─────────┘◀──── observa resultado ─│(ejecuta)  │
        │                              └───────────┘
        │ (repite hasta tener
        │  suficiente info)
        ▼
   Respuesta final
```

# Tools

### Creación de tool básica

In [ ]:
from langchain_core.tools import Tool
from langchain_experimental.utilities import PythonREPL

In [ ]:
python_repl = PythonREPL()

tool = Tool(
    name="python_repl",
    func=python_repl.run,
    description="Ejecuta codigo en Python en un interprete para calculos o logica matematica"
)

In [ ]:
output = tool.run("print(2 + 2)")
print(output)

### Creación de tool por medio del decorador

In [ ]:
from langchain_core.tools import tool

In [ ]:
# Definimos una herramienta personalizada utilizando el decorador @tool

@tool("Herramienta acceso base de datos usuarios.", return_direct=True)
def herramienta_personalizada(query: str) -> str:
    """Consulta la base de usuarios de la empresa."""
    # Codigo que accede a la basede datos
    return f"Respuesta a la consulta: {query}"

In [ ]:
output = herramienta_personalizada.run("Consulta de prueba")
print(output)

In [ ]:
print(f"Nombre de la herramienta: {herramienta_personalizada.name}")
print(f"Descripcion de la herramienta: {herramienta_personalizada.description}")

### Adicionando tool al LLM

In [1]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

import httpx
from dotenv import load_dotenv,find_dotenv

load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [2]:
# Asegurarse de que el modelo utilizado es compatible con herramientas personalizadas, como gpt-4o-mini o gpt-4o. En este ejemplo se utiliza gpt-4o-mini.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, http_client=http_client)

In [ ]:
# Definimos una herramienta personalizada utilizando el decorador @tool
@tool("user_db_tool")
def herramienta_personalizada(query: str) -> str:
    """Consulta la base de usuarios de la empresa."""
    # Codigo que accede a la basede datos
    return f"Respuesta a la consulta: {query}"

In [ ]:
# Creamos un LLM con herramientas personalizadas
llm_with_tools = llm.bind_tools([herramienta_personalizada])

In [ ]:
# Hacemos una consulta al LLM con herramientas personalizadas
response = llm_with_tools.invoke("Generea un resumen de la informacion que hay en la base de datos para el usuario con id 12345")

print(response)

content='' additional_kwargs={'tool_calls': [{'id': 'call_UdYBJIqnqln13BWZtWPc1kUA', 'function': {'arguments': '{"query":"SELECT * FROM users WHERE id = 12345"}', 'name': 'user_db_tool'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 69, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_956fb8d6db', 'id': 'chatcmpl-DzUHsrlGkxcyoLEAAROtBE74sqKbG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--019f43ac-73b1-78b3-98ea-08e6254751cb-0' tool_calls=[{'name': 'user_db_tool', 'args': {'query': 'SELECT * FROM users WHERE id = 12345'}, 'id': 'call_UdYBJIqnqln13BWZtWPc1kUA', 'type': 'tool_call'}] usage_metadata={'input_tokens': 69, 'output_tokens': 24, 

In [6]:
print(response.tool_calls[0])

{'name': 'user_db_tool', 'args': {'query': 'SELECT * FROM users WHERE id = 12345'}, 'id': 'call_UdYBJIqnqln13BWZtWPc1kUA', 'type': 'tool_call'}


In [ ]:
# Invocamos la herramienta personalizada con los argumentos obtenidos de la respuesta del LLM
tool_response = herramienta_personalizada.invoke(response.tool_calls[0]["args"])
print(tool_response)

Respuesta a la consulta: SELECT * FROM users WHERE id = 12345


### Adicionando tool al LLM y generando una cadena para su procesamiento

In [8]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from operator import attrgetter
from typing import Tuple

import httpx
from dotenv import load_dotenv,find_dotenv

In [9]:
load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [10]:
# Asegurarse de que el modelo utilizado es compatible con herramientas personalizadas, como gpt-4o-mini o gpt-4o. En este ejemplo se utiliza gpt-4o-mini.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, http_client=http_client)

In [11]:
# Definimos una herramienta personalizada utilizando el decorador @tool

@tool("user_db_tool", response_format="content_and_artifact")
def herramienta_personalizada(query: str) -> Tuple[str, dict]:
    """Consulta la base de usuarios de la empresa."""
    # Codigo que accede a la basede datos
    return f"Respuesta a la consulta: {query}", 10

In [12]:
# Creamos un LLM con herramientas personalizadas

llm_with_tools = llm.bind_tools([herramienta_personalizada])

In [13]:
# Creamos un chain que primero invoca el LLM con herramientas personalizadas y luego invoca la herramienta personalizada con los argumentos obtenidos de la respuesta del LLM
chain =  llm_with_tools | attrgetter("tool_calls") | herramienta_personalizada.map()

In [14]:
# Hacemos una consulta al chain
response = chain.invoke("Genera un resumen de la nformacion sobre el usuario con id 12345 que se encuentra en nuestra base de datos")

print(response)

[ToolMessage(content='Respuesta a la consulta: SELECT * FROM users WHERE id = 12345', name='user_db_tool', tool_call_id='call_TN3ORvgz1VKT1yXL1BkyfCgA', artifact=10)]
